# Датасет Московского метрополитена

In [ ]:
#!/bin/bash
!curl -L -o ~/Downloads/moscow-metro-stations.zip\
  https://www.kaggle.com/api/v1/datasets/download/samoilovmikhail/moscow-metro-stations

In [6]:
import pandas as pd

In [10]:
metro_lines_df = pd.read_csv(
    "data/moscow_metro/metro_lines.csv", 
    sep=",", 
    encoding="utf-8")

In [12]:
metro_stations_df = pd.read_csv(
    "data/moscow_metro/metro_stations.csv", 
    sep=",", 
    encoding="utf-8")

In [13]:
metro_stations_df

,Station_index,English_name,Line,Russian_name,Depth,Line_Neighbors,Transfers,Opened_Date,Lat,Lon,Station_Type
0,1,Bulvar Rokossovskogo,1,Бульвар Рокоссовского,-8.0,2,266,1990-08-01,55.8147,37.7342,column triple-span
1,2,Cherkizovskaya,1,Черкизовская,-9.0,1 3,267,1990-08-01,55.8039,37.7447,single-vault shallow
2,3,Preobrazhenskaya Ploshchad,1,Преображенская площадь,-8.0,2 4,NaN,1965-12-31,55.7964,37.7150,column triple-span
3,4,Sokolniki,1,Сокольники,-9.0,3 5,221,1935-05-15,55.7889,37.6803,single-vault shallow
4,5,Krasnoselskaya,1,Красносельская,-8.0,4 6,NaN,1935-05-15,55.7799,37.6673,column double-span
...,...,...,...,...,...,...,...,...,...,...,...
298,299,Nekrasovka,15,Некрасовка,-16.0,298,NaN,2019-06-03,55.7036,37.9264,column double-span
299,300,Novatorskaya,16,Новаторская,-11.0,301,235,2024-09-07,55.6690,37.5238,column triple-span
300,301,Universitet Druzhby Narodov,16,Университет Дружбы Народов,-23.0,300 302,NaN,2024-09-07,55.6484,37.5076,column triple-span
301,302,Generala Tyuleneva,16,Генерала Тюленева,-20.0,301 303,NaN,2024-09-07,55.6262,37.4860,column triple-span


In [14]:
import numpy as np

In [15]:
import pandas as pd
import networkx as nx

In [16]:
# Загрузка CSV-файла
df = metro_stations_df.copy()

In [18]:
# Приводим типы для уверенности
df['Line_Neighbors'] = df['Line_Neighbors'].astype(str)

In [19]:
# Создаем граф
G = nx.Graph()

In [20]:
# Добавим вершины
for index, row in df.iterrows():
    station = row['English_name']
    G.add_node(station)

In [21]:
# Добавим ребра по Line_Neighbors (связи по линии)
for _, row in df.iterrows():
    station = row['English_name']
    neighbors = row['Line_Neighbors'].split()
    
    for neighbor_index in neighbors:
        if neighbor_index.isdigit():
            neighbor_index = int(neighbor_index)
            neighbor_rows = df[df['Station_index'] == neighbor_index]
            if not neighbor_rows.empty:
                neighbor_station = neighbor_rows.iloc[0]['English_name']
                G.add_edge(station, neighbor_station)

In [22]:
# Вычисляем PageRank
pagerank_scores = nx.pagerank(G)

In [23]:
# Сортируем по убыванию значений PageRank
top_5 = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:5]

In [26]:
# Выводим результат
print("Топ-5 станций по PageRank:")
for i, (station, score) in enumerate(top_5, start=1):
    print(f"{i}. {station} — {score:.5f}")

Топ-5 станций по PageRank:
1. Kiyevskaya — 0.00771
2. Kuntsevskaya — 0.00750
3. Nizhegorodskaya — 0.00700
4. Kashirskaya — 0.00631
5. Prospekt Vernadskogo — 0.00617
